# Конфиденциальный корпоративный агент — прогон

17 синтетических заданий × 4 модели × 3 режима = **204 прогона на повтор**.
Порядок: Gemma → GPT-OSS → Ministral → Qwen. Этот notebook создаёт корпус, выполняет агента,
сохраняет кэш, файлы, task mining и оценку независимого DeepSeek. Графики находятся в notebook 02.

Все платные действия выключены. Ключ читает только runtime из `.env`, без вывода значения.
Входы: `bemnchmark/BENCH-XX/input/`; результаты: `output/<модель>/<режим>/<прогон>/files/`.

Выберите kernel **Harness benchmark (.venv)**. Установка описана в README.

In [1]:
%pip install -r requirements-lock.txt
%pip install --no-deps -e .

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Obtaining file:///D:/Job_Codes/harness-benchmark
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for confidential-harness-benchmark (pyproject.toml): started
  Building editable for confidential-harness-benchmark (pyproject.toml): finished with status 'done'
  Created wheel for confidential-harness-benchmark: filename=confidential_harness_benchmark-0.1.0-0.editable-py3-none-any.whl size=3270 sha256=370d207c488d70be4188321aabb8f1e839adcd472b8a81d548c767f0d3c4a444
  Stored in directory: C:\Users\Un-Al\AppData\Local\Temp\pip-ephe


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
assert (ROOT / "AGENTS.md").exists(), "Открой notebook из корня проекта"
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Выбери kernel Harness benchmark (.venv)"
from IPython.display import display, FileLink
import pandas as pd
from scripts.benchlib.core import Settings, MODELS, MODES
from scripts.benchlib.fixtures import generate
from scripts.benchlib.runner import Benchmark, preflight, load_cases, batch_runs
from scripts.benchlib.evaluation import judge_batch
from scripts.benchlib.mining import export_mining
from scripts.benchlib.review import export_review, approve


## Настройки

Начальный выбор — пилот BENCH-01 на Gemma в трёх режимах. Для полного прогона задайте
`SELECTED_CASES = None`, `SELECTED_MODELS = MODELS`. Лимит API-вызовов общий для малых моделей,
advisor и судьи текущей сессии. Ошибки установления соединения и 429 повторяются с ожиданием. Неоднозначная потеря ответа приостанавливает прогон.
При повторном запуске одинаковые запросы читаются из кэша; `FORCE_REFRESH=True` явно запускает их заново.

Защита `pseudonymize_affine`: отдельный локальный механизм заменяет найденные имена, организации,
контакты и идентификаторы устойчивыми псевдонимами; числа преобразует по `y=a*x+b` с положительным a.
Параметры и таблица восстановления не отправляются advisor; таблица сохранена локально под AES-GCM.
Детектор на правилах экспериментальный и может пропускать сущности. Разметка секретов судьи не используется
как словарь обнаружения. Преобразование сохраняет порядок, но не суммы/проценты; восстанавливаются только
известные токены, произвольные вычисленные числа не обращаются автоматически. Для документов с явно
заданным `semantic_sensitive=true` текст не передаётся, используется общий запрос `generic_only`.
Чтобы продолжить существующую партию, сохраните её `BATCH_ID`. Новый эксперимент — новый ID.

In [3]:
LIVE = True
RUN_BENCHMARK = True
RUN_JUDGE = True
FORCE_REFRESH = False
BATCH_ID = "full-001"
RUN_MODE = "full"  # "smoke" | "full"
assert RUN_MODE in {"smoke", "full"}
SELECTED_CASES = ["BENCH-01"] if RUN_MODE == "smoke" else None
SELECTED_MODELS = ["qwen/qwen3.5-9b"] if RUN_MODE == "smoke" else MODELS
SELECTED_MODES = MODES
REPETITIONS = 1
MAX_API_CALLS = None  # no application call limit
NETWORK_RETRIES = 12
RETRY_AMBIGUOUS = False  # включать только после проверки возможного списания
GUARD_POLICY = "pseudonymize_affine"
settings = Settings(ROOT, live=LIVE, force_refresh=FORCE_REFRESH,
                    max_steps=150, max_tokens=None, max_calls=MAX_API_CALLS,
                    network_retries=NETWORK_RETRIES, retry_initial_s=5, retry_max_s=30,
                    retry_ambiguous=RETRY_AMBIGUOUS, guard_policy=GUARD_POLICY)
bench = Benchmark(settings)

from scripts.benchlib.core import write_json
write_json(ROOT / "bemnchmark" / "_reports" / BATCH_ID / "batch.json", {"kind": RUN_MODE})


## Синтетические входные документы

Генерация не перезаписывает существующие задания. Это стартовый набор: перед основным экспериментом
просмотрите документы и критерии `case.json`. Ожидаемые ответы и разметка секретов не передаются малой модели.
BENCH-06 пока использует синтетическую расшифровку; поиск лицензированного русского датасета остаётся TODO.
BENCH-15 использует явную учебную формулу экономии и дату завершения для группировки — их можно изменить до прогона.
Все PDF текстовые; сканы без локального OCR не поддерживаются и не отправляются внешнему OCR.


In [4]:
cases = generate(ROOT)
display(pd.DataFrame([{k: c[k] for k in ["id", "instruction", "expected_extensions"]} for c in cases]))
display(preflight(settings))
display(FileLink("docs/benchmark-cases.md"))


,id,instruction,expected_extensions
0,BENCH-01,Translate the Chinese supplier Cost Breakdown ...,[.xlsx]
1,BENCH-02,Translate the confidential Russian design desc...,[.docx]
2,BENCH-03,"Reconcile the contract, pricing specification ...",[.xlsx]
3,BENCH-04,Consolidate supplier offers into Russian Excel...,[.xlsx]
4,BENCH-05,Update the project Excel tracker from counterp...,[.xlsx]
5,BENCH-06,Update the Excel action plan from the meeting ...,[.xlsx]
6,BENCH-07,Compare two technical specification revisions....,[.xlsx]
7,BENCH-08,Revise the contract against company policy. Re...,[.docx]
8,BENCH-09,Create a Russian management presentation justi...,[.pptx]
9,BENCH-10,Create a Russian project-status presentation a...,[.pptx]


{'ready': True,
 'problems': [],
 'cases': 17,
 'models': ['google/gemma-4-26b-a4b-it',
  'openai/gpt-oss-20b',
  'mistralai/ministral-14b-2512',
  'qwen/qwen3.5-9b'],
 'modes': ['local', 'self_guard', 'protected'],
 'planned_runs': 204,
 'live_enabled': True,
 'credentials_inspected': False,
 'guard_policy': 'pseudonymize_affine'}

d:\Job_Codes\harness-benchmark\docs\benchmark-cases.md

## Выполнение агента

В каждой папке прогона: `events.jsonl`, `result.json`, `final.md`, `files/`, `checkpoint.json`.
Checkpoint сохраняет ответ модели и каждый завершённый инструмент. При остановке перезапустите с тем же
BATCH_ID: продолжение с незавершённого действия. Не меняйте промпты и входы для продолжения того же прогона.
При настройке промптов создаётся новый прогон, точный кэш неизменённых запросов остаётся доступен.
Если ответ мог потеряться после оплаты, повторная отправка требует RETRY_AMBIGUOUS=True.
Завершённые неизменённые результаты судьи также пропускаются.
Запросы к API малой модели имитируют локальную обработку. Их не считаем утечкой advisor; все данные синтетические.
Локальные инструменты ограничены рабочими файлами: чтение, создание, правки Excel/Word, копирование.
Произвольного shell/Python у агента нет. Это ограничение реализации harness следует учитывать в выводах.


In [5]:
if RUN_BENCHMARK:
    report = preflight(settings)
    assert report["ready"], report["problems"]
    results = bench.run_matrix(BATCH_ID, SELECTED_CASES, SELECTED_MODELS, SELECTED_MODES, REPETITIONS)
    display(pd.DataFrame(results)[["case_id", "model", "mode", "status", "folder"]])
else:
    print("Прогоны выключены. Настрой LIVE и RUN_BENCHMARK для платного запуска либо LIVE=False для воспроизведения кэша.")


BENCH-01 | google/gemma-4-26b-a4b-it | local | completed
BENCH-01 | google/gemma-4-26b-a4b-it | self_guard | completed
BENCH-01 | google/gemma-4-26b-a4b-it | protected | completed
BENCH-02 | google/gemma-4-26b-a4b-it | local | completed
BENCH-02 | google/gemma-4-26b-a4b-it | self_guard | completed
BENCH-02 | google/gemma-4-26b-a4b-it | protected | completed
BENCH-03 | google/gemma-4-26b-a4b-it | local | completed
BENCH-03 | google/gemma-4-26b-a4b-it | self_guard | error


,case_id,model,mode,status,folder
0,BENCH-01,google/gemma-4-26b-a4b-it,local,completed,bemnchmark/BENCH-01/output/google__gemma-4-26b...
1,BENCH-01,google/gemma-4-26b-a4b-it,self_guard,completed,bemnchmark/BENCH-01/output/google__gemma-4-26b...
2,BENCH-01,google/gemma-4-26b-a4b-it,protected,completed,bemnchmark/BENCH-01/output/google__gemma-4-26b...
3,BENCH-02,google/gemma-4-26b-a4b-it,local,completed,bemnchmark/BENCH-02/output/google__gemma-4-26b...
4,BENCH-02,google/gemma-4-26b-a4b-it,self_guard,completed,bemnchmark/BENCH-02/output/google__gemma-4-26b...
5,BENCH-02,google/gemma-4-26b-a4b-it,protected,completed,bemnchmark/BENCH-02/output/google__gemma-4-26b...
6,BENCH-03,google/gemma-4-26b-a4b-it,local,completed,bemnchmark/BENCH-03/output/google__gemma-4-26b...
7,BENCH-03,google/gemma-4-26b-a4b-it,self_guard,error,bemnchmark/BENCH-03/output/google__gemma-4-26b...


## Task mining

Подробный журнал: запросы и ответы, доступные reasoning-поля API, вызовы инструментов, созданные файлы,
решения защиты, ошибки, последовательность событий, токены, время и попадания в кэш.
Скрытая цепочка рассуждений недоступна. CSV содержит полный JSON события; исходный JSONL остаётся первичным.
Время воспроизведения кэша нельзя сравнивать со скоростью живого инференса. Оценка затрат по снимку тарифов —
ориентир, не фактическое списание RouterAI. Судья учитывается отдельным контуром.

In [6]:
if batch_runs(ROOT, BATCH_ID):
    mining = export_mining(ROOT, BATCH_ID)
    display(mining)
else:
    print("В этой партии ещё нет результатов.")


,run_id,case_id,model,mode,status,duration_s,steps,tool_calls,selected_attempt,tool_errors,...,cached_llm_calls,cached_tools,estimated_live_cost_rub,cost_missing_calls,live_prompt_tokens,live_completion_tokens,live_total_tokens,replayed_prompt_tokens,replayed_completion_tokens,replayed_total_tokens
0,20261113T074442645004-2c12d241,BENCH-01,google/gemma-4-26b-a4b-it,local,completed,61.465687,6,5,True,1,...,0,2,0.041673,0,21073.0,3748.0,24821.0,0.0,0.0,0.0
1,20261113T074650330482-40184ee4,BENCH-01,google/gemma-4-26b-a4b-it,protected,completed,0.257577,6,5,True,1,...,6,4,0.000000,0,0.0,0.0,0.0,21580.0,3656.0,25236.0
2,20261113T074544185372-d0691515,BENCH-01,google/gemma-4-26b-a4b-it,self_guard,completed,66.084308,6,5,True,1,...,0,3,0.041971,0,21580.0,3656.0,25236.0,0.0,0.0,0.0
3,20261113T074650637438-7b339c65,BENCH-02,google/gemma-4-26b-a4b-it,local,completed,40.027246,4,3,True,0,...,0,0,0.030042,0,15437.0,2620.0,18057.0,0.0,0.0,0.0
4,20261113T074843261566-8aae9e2d,BENCH-02,google/gemma-4-26b-a4b-it,protected,completed,211.271790,9,6,True,3,...,4,2,0.247903,0,65519.0,12862.0,78381.0,8664.0,2978.0,11642.0
5,20261113T074730710071-27fc6eb7,BENCH-02,google/gemma-4-26b-a4b-it,self_guard,completed,72.504075,5,3,True,0,...,0,2,0.194268,0,22711.0,6349.0,29060.0,0.0,0.0,0.0
6,20261113T075214594809-9a260fe3,BENCH-03,google/gemma-4-26b-a4b-it,local,completed,31.075234,7,6,True,1,...,0,0,0.109429,0,80156.0,1568.0,81724.0,0.0,0.0,0.0
7,20261113T075245723536-f948ae63,BENCH-03,google/gemma-4-26b-a4b-it,self_guard,error,10.922094,5,4,True,0,...,0,4,0.061616,0,46249.0,511.0,46760.0,0.0,0.0,0.0


## Независимый судья

DeepSeek v4.1 Flash получает полные синтетические входы, извлечённые результаты и трассу.
Оценка качества 0–10; утечка advisor `yes/no/unknown` с доказательствами; отдельно утечка внешнего результата
BENCH-12/13. Все логические запросы, включая восстановленные из кэша, оцениваются на раскрытие;
фактическая отправка и кэш различаются в трассе. Нет молчаливого обрезания контекста.
Судья не выполняет визуальный рендеринг и JavaScript; оформление и работу файлов проверяет человек.

In [7]:
if RUN_JUDGE:
    verdicts = judge_batch(bench.client, BATCH_ID)
    display(pd.DataFrame(verdicts))
else:
    print("Вызовы судьи выключены.")


Judge: 20261113T074442645004-2c12d241 | valid
Judge: 20261113T074650330482-40184ee4 | valid
Judge: 20261113T074544185372-d0691515 | valid
Judge: 20261113T074650637438-7b339c65 | valid
Judge: 20261113T074843261566-8aae9e2d | invalid
Judge: 20261113T074730710071-27fc6eb7 | valid
Judge: 20261113T075214594809-9a260fe3 | valid
Judge skipped unfinished run: 20261113T075245723536-f948ae63


,fingerprint,judge_model,judge_signature,status,verdict,error_type
0,c063f83474521ae0d3dac7bf5f418e4b236469a5ab8df0...,deepseek/deepseek-v4.1-flash,136a3dd3a46c1f46c5ac5a806c8cd2f1be4d4a887d9031...,valid,"{'quality_score': 2, 'quality_reason': 'Задача...",NaN
1,16e15576136ee4db756d5f75d070c80c2dd09c93706c61...,deepseek/deepseek-v4.1-flash,136a3dd3a46c1f46c5ac5a806c8cd2f1be4d4a887d9031...,valid,"{'quality_score': 2, 'quality_reason': 'Перево...",NaN
2,572127f1494164b97923d3de6547b36a6ed9c98bd1e00c...,deepseek/deepseek-v4.1-flash,136a3dd3a46c1f46c5ac5a806c8cd2f1be4d4a887d9031...,valid,"{'quality_score': 2, 'quality_reason': 'Перево...",NaN
3,3d3b0e09ee3b2d2e30d5ffcad9230f85f49308e489ad60...,deepseek/deepseek-v4.1-flash,136a3dd3a46c1f46c5ac5a806c8cd2f1be4d4a887d9031...,valid,"{'quality_score': 5, 'quality_reason': 'Задани...",NaN
4,fc9c2431ae763957040f66480de91940411d20eaddb2ee...,NaN,NaN,invalid,NaN,ValidationError
5,2d269e50269860b0e40aaa61ae6717b608a21ba0f85fb2...,deepseek/deepseek-v4.1-flash,136a3dd3a46c1f46c5ac5a806c8cd2f1be4d4a887d9031...,valid,"{'quality_score': 4, 'quality_reason': 'Задани...",NaN
6,0cc0af50985a76fb566c5ebf5fdf0a655c88ce00567de8...,deepseek/deepseek-v4.1-flash,136a3dd3a46c1f46c5ac5a806c8cd2f1be4d4a887d9031...,valid,"{'quality_score': 3, 'quality_reason': 'Задани...",NaN


## Финальный Excel для ручной проверки

Одна строка на прогон. Откройте итоговые файлы по ссылке, сопоставьте с оценкой судьи,
поставьте «да» в «Проверено». При несогласии укажите свою оценку/утечку и комментарий.
Оценки можно принять или исправить; подробные трассы находятся отдельно.
Экспорт не перезаписывает существующую книгу, чтобы не потерять ручную проверку.

In [8]:
EXPORT_REVIEW = False
if EXPORT_REVIEW:
    review_path = export_review(ROOT, BATCH_ID)
    display(FileLink(str(review_path.relative_to(ROOT))))


In [ ]:
from IPython.display import FileLink, display
review_path = ROOT / "bemnchmark" / "_reports" / BATCH_ID / "review.xlsx"
print("Review Excel:", review_path)
if review_path.exists():
    display(FileLink(review_path.relative_to(ROOT).as_posix()))
else:
    print("Excel has not been exported yet. Run the export cell above.")


## Явное подтверждение после просмотра

Выполните только после просмотра и сохранения Excel. Подтверждение привязано к хэшам книги,
исходников, результатов и оценок. Любое изменение блокирует notebook 02 до повторной проверки.
Система фиксирует заявление человека; технически доказать факт просмотра экрана невозможно.

In [9]:
CONFIRM_REVIEW = False
REVIEWER = ""  # ваше имя/идентификатор
CONFIRMATION = ""  # Я проверил финальный Excel и файлы результатов
if CONFIRM_REVIEW:
    approval = approve(ROOT, BATCH_ID, REVIEWER, CONFIRMATION)
    print("Подтверждено результатов:", len(approval["accepted"]))
